# 03. Аналитика для диагностики

Расчёты по обработанным слоям (`data/processed/`) и границам версий (ноутбук 01):

1. Баланс территории v2 (жилая / нежилая / промышленная / зелень / прочее);
2. Застройка по периодам (HOITH) — возрастная структура;
3. Пешая доступность базовых сервисов (школы, детсады, здоровье) — буферный анализ по прямой с оговоркой (сетевой расчёт — этап IduEdu);
4. «Отрыв берега»: доля береговой полосы Невы и Охты, занятая промышленностью/огороженными территориями — ключевой конфликт района.

Все числа диагностического документа (`docs/diagnostics.md`) проверяются `scripts/check_numbers.py`.

In [1]:
# Настройка и загрузка слоёв
from pathlib import Path

import geopandas as gpd
import pandas as pd

PROJECT = Path.cwd().resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
RAW, PROC = PROJECT / "data/raw", PROJECT / "data/processed"
CRS_METRIC = 32636

versions = gpd.read_file(PROC / "boundary_versions.geojson").set_index("version")
V2 = versions.loc["v2", "geometry"]
V3 = versions.loc["v3", "geometry"]
AREA_V2 = versions.loc["v2", "area_km2"]
AREA_V3 = versions.loc["v3", "area_km2"]
print(f"v2 {AREA_V2:.2f} км2 | v3 {AREA_V3:.2f} км2")

building = gpd.read_file(PROC / "building.geojson")
green = gpd.read_file(PROC / "green.geojson")
services = gpd.read_file(PROC / "services.geojson")
roads = gpd.read_file(PROC / "roads.geojson")
water = gpd.read_file(PROC / "water.geojson")
house_age = gpd.read_file(PROC / "house_age.geojson")
okn = gpd.read_file(PROC / "okn.geojson")
print("слои загружены")

v2 5.26 км2 | v3 11.06 км2
слои загружены


/home/mfthesun/Projects/malaya-okhta-strategy/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: Several features with id = 2305221 have been found. Altering it to be unique. This warning will not be emitted anymore for this layer
  return ogr_read(


In [3]:
# Баланс территории v2: что занимает землю района (без воды)
land = V2.difference(water.union_all().intersection(V2).buffer(0))
land_km2 = gpd.GeoSeries([land], crs=4326).to_crs(CRS_METRIC).area.iloc[0] / 1e6

def share_km2(gdf, mask_geom):
    """Суммарная площадь пересечений слоя с территорией, км2 (без наложений внутри слоя)."""
    if not len(gdf):
        return 0.0
    clipped = gpd.GeoSeries(gdf.geometry, crs=4326).intersection(mask_geom)
    clipped = clipped[~clipped.is_empty]
    if not len(clipped):
        return 0.0
    merged = clipped.union_all()
    return gpd.GeoSeries([merged], crs=4326).to_crs(CRS_METRIC).area.iloc[0] / 1e6

# классификация застройки
LIVING = ["apartments", "residential", "dormitory", "detached", "house", "semidetached_house", "yes"]
IND = ["industrial", "warehouse"]
building["kind"] = "прочая"
building.loc[building.building.isin(IND), "kind"] = "промышленно-складская"
building.loc[building.building.isin(LIVING), "kind"] = "жилая"

rows = {}
for kind in ["жилая", "промышленно-складская", "прочая"]:
    rows[kind] = share_km2(building[building.kind == kind], V2)
GREEN_TAGS = ["forest", "wood", "grass", "grassland", "meadow", "park", "scrub", "recreation_ground"]
rows["зелень"] = share_km2(green[green.get("landuse").isin(GREEN_TAGS) | green.get("natural", pd.Series(dtype=str)).isin(GREEN_TAGS)], V2)
water_v2 = share_km2(water, V2)

print(f"Территория v2 (без воды): {land_km2:.2f} км2, вода: {water_v2:.2f} км2")
for k, v in rows.items():
    print(f"  {k}: {v:.2f} км2 ({100 * v / AREA_V2:.1f} % территории округа)")
print(f"  покрыто всеми слоями: {sum(rows.values()):.2f} км2")

Территория v2 (без воды): 5.16 км2, вода: 0.09 км2
  жилая: 0.46 км2 (8.7 % территории округа)
  промышленно-складская: 0.03 км2 (0.5 % территории округа)
  прочая: 0.26 км2 (4.9 % территории округа)
  зелень: 0.30 км2 (5.7 % территории округа)
  покрыто всеми слоями: 1.04 км2
